In [1]:
from vizdoom import *
import cv2
import numpy as np
import os

In [2]:
# Gym imports
import gymnasium as gym
from gymnasium import Env
from gymnasium.spaces import Discrete, Box

from stable_baselines3.common import env_checker

In [3]:
# Create Vizdoom OpenAI Gym Environment
class VizDoomGym(Env): 
    def __init__(self, render=False): 
        super().__init__()

        # Setup the game 
        self.game = DoomGame()
        cfg_path ="/home/acaia/AgenticTakeCover/AgenticTakeCover/github/ViZDoom/scenarios/take_cover.cfg"
        self.game.load_config(cfg_path)
        
        # Render frame logic
        self.game.set_window_visible(render)
        
        # Start the game 
        self.game.init()
        
        # Create the action space and observation space
        self.observation_space = Box(low=0, high=255, shape=(100,160,1), dtype=np.uint8) 
        self.action_space = Discrete(2) # Move left, move right
        
    # This is how we take a step in the environment
    def step(self, action):
        # Specify action and take step 
        actions = np.identity(2, dtype=np.uint8)
        reward = self.game.make_action(actions[action].tolist(), 4) 
        
        # Get the new state of the game and check if it's done
        if self.game.get_state(): 
            state = self.game.get_state().screen_buffer
            state = self.grayscale(state)
            health = self.game.get_state().game_variables[0]
            info = {"health": health}
        else: 
            state = np.zeros(self.observation_space.shape, dtype=np.uint8)
            info = {"health": 0}
        
        terminated = self.game.is_episode_finished()
        truncated = False

        return state, reward, terminated, truncated, info
    
    # Define how to render the game or environment 
    def render(self): 
        pass
    
    # Starting a new game 
    def reset(self, seed=None, options=None): 
        super().reset(seed=seed)
        self.game.new_episode()
        state = self.game.get_state().screen_buffer
        obs = self.grayscale(state)
        info = {}
        return obs, info
    
    # Grayscale the game frame and resize it 
    def grayscale(self, observation):
        gray = cv2.cvtColor(np.moveaxis(observation, 0, -1), cv2.COLOR_BGR2GRAY)
        resize = cv2.resize(gray, (160,100), interpolation=cv2.INTER_CUBIC)
        state = np.reshape(resize, (100,160,1))
        return state
    
    # Call to close down the game
    def close(self): 
        self.game.close()

In [4]:
# Sanity check 
env = VizDoomGym(render=False)
env_checker.check_env(env)
env.close()

In [5]:
# Setting up callbacks
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.monitor import Monitor

In [6]:
class TrainAndLoggingCallback(BaseCallback):

    def __init__(self, check_freq, save_path, verbose=1):
        super(TrainAndLoggingCallback, self).__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path

    def _init_callback(self):
        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            model_path = os.path.join(self.save_path, 'best_model_{}'.format(self.n_calls))
            self.model.save(model_path)

        return True

In [7]:
CHECKPOINT_DIR = './train/train_take_cover'
LOG_DIR = './logs/log_take_cover'
MODEL_DIR = './models/take_cover_ppo_final'

In [8]:
callback = TrainAndLoggingCallback(check_freq=10000, save_path=CHECKPOINT_DIR)

#### Training the model

In [9]:
from stable_baselines3 import PPO

In [10]:
env = VizDoomGym(render=False)

In [11]:
model = PPO(
    policy = 'CnnPolicy',
    env = env,
    tensorboard_log=LOG_DIR,
    verbose=1,
    learning_rate=1e-4,
    n_steps=4096,
    batch_size=64,
    n_epochs=4,
    ent_coef=0.01
)

/home/acaia/AgenticTakeCover/AgenticTakeCover/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.


In [12]:
model.learn(total_timesteps=500000, callback=callback)

Logging to ./logs/log_take_cover/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 76.9     |
|    ep_rew_mean     | 306      |
| time/              |          |
|    fps             | 163      |
|    iterations      | 1        |
|    time_elapsed    | 25       |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 73          |
|    ep_rew_mean          | 290         |
| time/                   |             |
|    fps                  | 120         |
|    iterations           | 2           |
|    time_elapsed         | 67          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.010091718 |
|    clip_fraction        | 0.14        |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.682      |
|    explained_variance   | -0.00

In [13]:
model.save(MODEL_DIR)
env.close()

/home/acaia/AgenticTakeCover/AgenticTakeCover/.venv/lib/python3.10/site-packages/stable_baselines3/common/save_util.py:284: UserWarning: Path 'models' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


#### Testing th model